<p class="hz-kicker">Week 4 · Self-paced lab</p>

# An overbooking policy

The operations manager wants a rule for overbooking that she can defend: which rooms to resell, at which hotel, and how many rooms to oversell for a busy week. You will set it with this week's model and the costs of each mistake, then write her a short memo.

<ul class="hz-lesson-meta"><li>≈ 2 hours</li><li>5 tasks</li><li>Runs in your browser</li><li>Graded quiz</li></ul>

**What we saw in the live session**

- A **logistic regression** gives every booking a probability of cancelling. On the test bookings it ranks cancellations above arrivals three times in four (AUC 0.74).
- A **threshold** turns the probability into a decision: resell the room or not. Lower thresholds catch more cancellations (recall) and walk more guests (lower precision).
- The cheapest threshold comes from the two **costs**: reselling pays off when p is above walk / (walk + empty room).

**Setup.** Run this cell first. You don't need to read it: it loads the bookings, retrains the live session's model (same features, same stratified split) and defines two helpers, `policy` and `cheapest`, that count walked guests and empty rooms for a threshold.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score

DATA = "https://raw.githubusercontent.com/hizocar/mba-business-analytics-1/main/docs/datasets/"
sns.set_theme(style="whitegrid")
usd = lambda x: f"USD {x:,.0f}"

df = pd.read_csv(DATA + "altamar_bookings_clean.csv")
y = df["canceled"]
X = df.drop(columns=["booking_id", "booking_date", "arrival_date", "canceled"])
X["arrival_month"] = pd.to_datetime(df["arrival_date"]).dt.month
BASELINE = {"hotel": ["Santiago", "Viña del Mar", "Puerto Varas"], "room_type": ["Standard", "Superior", "Suite"],
            "customer_type": ["Business", "Leisure", "Group"], "deposit": ["No deposit", "Non-refundable"],
            "channel": ["Direct", "Online travel agency", "Corporate", "Travel agent"],
            "country": ["Chile", "Argentina", "Brazil", "United States", "Europe", "Other", "Unknown"]}
for col, order in BASELINE.items():
    X[col] = pd.Categorical(X[col], order)
X = pd.get_dummies(X, columns=list(BASELINE), drop_first=True, dtype="int64")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
model = Pipeline([("scale", StandardScaler()), ("logit", LogisticRegression(max_iter=2000))]).fit(X_train, y_train)
test = df.loc[X_test.index].assign(p_cancel=model.predict_proba(X_test)[:, 1])

def policy(p, cancelled, threshold, cost_empty, cost_walk):
    resold = p >= threshold
    walked, empty = int((resold & (cancelled == 0)).sum()), int((~resold & (cancelled == 1)).sum())
    return {"resold": int(resold.sum()), "walked": walked, "empty": empty, "cost": cost_empty * empty + cost_walk * walked}

def cheapest(p, cancelled, cost_empty, cost_walk):
    rows = pd.DataFrame([{"threshold": t, **policy(p, cancelled, t, cost_empty, cost_walk)}
                         for t in np.round(np.arange(0, 1.001, 0.01), 2)])
    best = rows.loc[rows["cost"].idxmin()]
    return {"threshold": best["threshold"], **{k: int(best[k]) for k in ["resold", "walked", "empty", "cost"]}}

print(f"Model retrained on {len(X_train):,} bookings; {len(test):,} test bookings scored.")

Model retrained on 4,473 bookings; 1,491 test bookings scored.


## <span class="hz-step-n">01</span> Your costs, your threshold

Every policy has two mistakes: a cancellation we did not resell (**an empty room**) and a resold booking whose guest comes anyway (**a walked guest**). Start in the simulator: move both costs and the threshold, and watch the cheapest point move.

<div class="hz-widget" data-widget="cost-threshold">
<p>▶ Interactive simulator: <a href="https://hizocar.github.io/mba-business-analytics-1/sessions/Session_4_Lab/">open this session on the course website</a> to use it.</p>
</div>

Now in code. ✏️ Set the two costs, run, and read the cheapest threshold, the bookings it flags, and what it saves against doing nothing.

In [2]:
# ✏️ Change these settings
COST_EMPTY_ROOM = 400   # USD: a cancelled booking we did not resell, so the room stays empty
COST_WALK = 300         # USD: a resold booking whose guest comes: another hotel, taxi, voucher

p, cancelled = test["p_cancel"].values, test["canceled"].values
best_all = cheapest(p, cancelled, COST_EMPTY_ROOM, COST_WALK)
nothing_all = COST_EMPTY_ROOM * cancelled.sum()
print(f"Cheapest threshold: {best_all['threshold']:.2f} (break-even {COST_WALK / (COST_WALK + COST_EMPTY_ROOM):.2f})")
print(f"Bookings flagged, rooms resold: {best_all['resold']} of {len(p):,}")
print(f"  guests walked {best_all['walked']}, rooms left empty {best_all['empty']}")
print(f"Expected cost {usd(best_all['cost'])} against {usd(nothing_all)} doing nothing: "
      f"{usd(nothing_all - best_all['cost'])} saved ({(nothing_all - best_all['cost']) / nothing_all:.0%})")

Cheapest threshold: 0.43 (break-even 0.43)
Bookings flagged, rooms resold: 451 of 1,491
  guests walked 189, rooms left empty 241
Expected cost USD 153,100 against USD 201,200 doing nothing: USD 48,100 saved (24%)


**Think about it.** Guest relations says a walked guest also costs a bad review: set `COST_WALK = 600`. Then go back to 300 and set `COST_EMPTY_ROOM = 130`, a single night at the average rate. What happens to the threshold and to the savings each time? Why?

<details><summary>Show the answer</summary>

With a walk at USD 600 the break-even rises to 600 / 1,000 = 0.6 and the cheapest threshold moves to 0.54: only 260 rooms are resold and the saving falls from USD 48,100 to about USD 7,000. Dearer walks mean fewer, surer resales.

With an empty room at only USD 130 the break-even is 300 / 430 ≈ 0.70, and the cheapest policy resells 2 rooms for a saving of USD 260: when an empty room is cheap, overbooking is not worth the risk. The model is the same every time; only the money changed, and with it the decision.

</details>

## <span class="hz-step-n">02</span> One policy per hotel

The three hotels are different businesses. An empty room in Puerto Varas loses a long, expensive stay; in Santiago, a short business night. Walking a guest in Santiago is easy (dozens of hotels nearby); in Puerto Varas, in high season, it can mean a taxi to the next town. ✏️ Set the two costs for each hotel and compare the policies.

In [3]:
# ✏️ Change these settings: (empty room, walked guest) in USD
COSTS = {"Santiago":     (250, 200),     # short business stays; many hotels nearby
         "Viña del Mar": (480, 300),
         "Puerto Varas": (560, 450)}     # long, dear stays; few comparable hotels nearby

rows = []
for hotel, (empty, walk) in COSTS.items():
    h = test[test["hotel"] == hotel]
    b = cheapest(h["p_cancel"].values, h["canceled"].values, empty, walk)
    rows.append({"hotel": hotel, "test bookings": len(h), "cancel rate": h["canceled"].mean(),
                 "break-even": walk / (walk + empty), "cheapest threshold": b["threshold"],
                 "rooms resold": b["resold"], "guests walked": b["walked"],
                 "saved vs doing nothing": empty * h["canceled"].sum() - b["cost"]})
by_hotel = pd.DataFrame(rows).round(2)
by_hotel

,hotel,test bookings,cancel rate,break-even,cheapest threshold,rooms resold,guests walked,saved vs doing nothing
0,Santiago,591,0.25,0.44,0.43,85,40,3250
1,Viña del Mar,526,0.42,0.38,0.36,293,122,45480
2,Puerto Varas,374,0.36,0.45,0.49,99,38,17060


**Think about it.** Which hotel gains most from the policy, and why that one? Would you use a single company-wide threshold or one per hotel? What is the risk of setting a threshold on only a few hundred bookings?

<details><summary>Show the answer</summary>

**Viña del Mar** gains most, about USD 45,000: the highest cancellation rate (42%) and dear empty rooms give it the lowest threshold (0.36). **Santiago** saves only about USD 3,000: few cancellations, cheap rooms, and a walk costs almost as much as an empty room.

Different costs justify **one threshold per hotel**. The risk: with a few hundred bookings, the cheapest threshold jumps around by chance (Viña's 0.36 versus its break-even of 0.38). Anchor each threshold on its break-even, and check it on fresh validation data before using it.

</details>

## <span class="hz-step-n">03</span> How many rooms for a busy week?

Operations does not resell rooms one by one; it decides **how many rooms to oversell** for a week. The model helps: adding up the probabilities of all bookings for that week gives the **expected number of cancellations**, just as ten bookings with a 30% risk each should bring about three cancellations. ✏️ Choose a hotel and a week.

In [4]:
# ✏️ Change these settings
HOTEL = "Viña del Mar"
WEEK_START = "2025-02-10"     # a Monday; the week runs to Sunday
MAX_WALK_RISK = 0.10          # the highest chance of walking at least one guest we accept

days_from_start = (pd.to_datetime(df["arrival_date"]) - pd.Timestamp(WEEK_START)).dt.days
week = (df["hotel"] == HOTEL) & days_from_start.between(0, 6)
p_week = model.predict_proba(X[week])[:, 1]
expected = p_week.sum()
print(f"{week.sum()} bookings arrive at {HOTEL} in the week of {WEEK_START}")
print(f"Expected cancellations (sum of the probabilities): {expected:.1f}")
print(f"What really happened: {df.loc[week, 'canceled'].sum()} cancellations")

54 bookings arrive at Viña del Mar in the week of 2025-02-10
Expected cancellations (sum of the probabilities): 24.2
What really happened: 24 cancellations


Overselling as many rooms as the expected cancellations is a coin flip: in about half of the weeks fewer guests cancel and someone is walked. We simulate 10,000 versions of this week and, for each number of rooms oversold, count the chance of walking someone and the expected cost (with the costs of task 1).

**Watch out:** most of these bookings helped train the model, so this is a demonstration; in real use you score the bookings on the books for next week. We also assume the hotel is full, so every oversold room is a walk if its guest comes.

In [5]:
rng = np.random.default_rng(42)
cancellations = (rng.random((10_000, len(p_week))) < p_week).sum(axis=1)   # 10,000 simulated weeks
rows = []
for k in range(max(0, round(expected) - 8), round(expected) + 4):
    walked, empty = np.maximum(k - cancellations, 0), np.maximum(cancellations - k, 0)
    rows.append({"rooms oversold": k, "chance of walking someone": (cancellations < k).mean(),
                 "expected walks": walked.mean(), "expected cost": COST_WALK * walked.mean() + COST_EMPTY_ROOM * empty.mean()})
plan = pd.DataFrame(rows)
safe = plan.loc[plan["chance of walking someone"] <= MAX_WALK_RISK, "rooms oversold"].max()
print(f"Oversell {safe} rooms to keep the chance of walking anyone under {MAX_WALK_RISK:.0%}; "
      f"the cheapest on average is {plan.loc[plan['expected cost'].idxmin(), 'rooms oversold']}.")
plan.round(2)

Oversell 20 rooms to keep the chance of walking anyone under 10%; the cheapest on average is 25.


,rooms oversold,chance of walking someone,expected walks,expected cost
0,16,0.00,0.01,3301.58
1,17,0.01,0.01,2907.67
2,18,0.02,0.04,2522.16
3,19,0.04,0.08,2150.16
4,20,0.08,0.15,1802.73
5,21,0.13,0.28,1493.73
6,22,0.20,0.48,1237.02
7,23,0.30,0.79,1048.84
8,24,0.42,1.20,940.18
9,25,0.53,1.74,913.63


**Think about it.** Why is the "safe" number lower than the expected cancellations? Try `MAX_WALK_RISK = 0.02` and `0.30`. Then try a quiet week, `WEEK_START = "2025-06-09"`. Who in the company should choose `MAX_WALK_RISK`?

<details><summary>Show the answer</summary>

Cancellations vary from week to week around their expected value (24 here). Overselling 24 rooms walks someone in about four weeks out of ten; to be 90% sure of walking nobody you oversell fewer, 20. With `0.02` you oversell 17; with `0.30`, 22. The cheapest on average (25) accepts a walk in half of the weeks, because an empty room costs more than a walk.

A quiet June week has 11 bookings and 3.6 expected cancellations, and the hotel is probably not full: little reason to oversell.

`MAX_WALK_RISK` is a guest-experience decision for the general manager, not for the analyst.

</details>

## <span class="hz-step-n">04</span> Precision or recall?

For a resale decision, **precision** is the share of resold rooms whose guest really cancels (every miss is a walked guest), and **recall** is the share of cancellations whose room we resell (every miss is an empty room). The simulator shows the trade-off on 80 real test bookings.

<div class="hz-widget" data-widget="threshold">
<p>▶ Interactive simulator: <a href="https://hizocar.github.io/mba-business-analytics-1/sessions/Session_4_Lab/">open this session on the course website</a> to use it.</p>
</div>

✏️ Try thresholds between 0.3 and 0.7 and read both metrics, in rooms and guests.

In [6]:
# ✏️ Change these settings
THRESHOLD = 0.43

flag = p >= THRESHOLD
print(f"Threshold {THRESHOLD:.2f}: {flag.sum()} of {len(p):,} bookings flagged, their rooms resold")
print(f"Precision {precision_score(cancelled, flag):.2f}: share of resold rooms whose guest really cancels")
print(f"Recall    {recall_score(cancelled, flag):.2f}: share of cancellations whose room we resell")
print(f"→ {(flag & (cancelled == 0)).sum()} guests walked, {(~flag & (cancelled == 1)).sum()} rooms left empty")

Threshold 0.43: 451 of 1,491 bookings flagged, their rooms resold
Precision 0.58: share of resold rooms whose guest really cancels
Recall    0.52: share of cancellations whose room we resell
→ 189 guests walked, 241 rooms left empty


**Think about it.** For Altamar's overbooking decision, which matters more, precision or recall? Does your answer depend on the hotel? What would make you change your mind?

<details><summary>Show the answer</summary>

Usually **precision**: every false flag is a guest turned away at the front desk, which costs more than money (reviews, loyalty). A missed cancellation only leaves a room empty, which is what happens today anyway.

But the honest answer is "it depends on the costs". Where an empty room costs much more than a walk (Viña in high season), recall matters more and the threshold goes down; where walking a guest is hard (Puerto Varas), precision matters more and it goes up. You would change your mind if the costs changed, for example if a partner hotel next door made walks cheap.

</details>

## <span class="hz-step-n">05</span> A memo to the operations manager

The cell drafts a memo with your numbers from tasks 1 to 3. Run it, then read it as the operations manager would: is the recommendation clear in the first line? Are the risks honest?

In [7]:
memo = f"""To: Operations manager, Altamar Hotels
Subject: An overbooking rule based on each booking's cancellation risk

1. Recommendation. Resell the room of every booking with a predicted risk of cancelling of
   {best_all['threshold']:.0%} or more. On {len(p):,} past bookings the model had never seen, this would have
   cost {usd(best_all['cost'])} instead of {usd(nothing_all)}: {usd(nothing_all - best_all['cost'])} saved.
2. The price. It resells {best_all['resold']} rooms and walks {best_all['walked']} guests. Each walk is costed at {usd(COST_WALK)}.
3. Per hotel. Cheapest thresholds: {', '.join(f"{r['hotel']} {r['cheapest threshold']:.2f}" for _, r in by_hotel.iterrows())}.
4. Next busy week. {HOTEL}, week of {WEEK_START}: about {expected:.0f} cancellations expected; overselling
   {safe} rooms keeps the chance of walking anyone under {MAX_WALK_RISK:.0%}.
5. Caveats. The model ranks bookings well, not perfectly (AUC 0.74), and it overestimates the risk
   of bookings made more than four months ahead."""
print(memo)

To: Operations manager, Altamar Hotels
Subject: An overbooking rule based on each booking's cancellation risk

1. Recommendation. Resell the room of every booking with a predicted risk of cancelling of
   43% or more. On 1,491 past bookings the model had never seen, this would have
   cost USD 153,100 instead of USD 201,200: USD 48,100 saved.
2. The price. It resells 451 rooms and walks 189 guests. Each walk is costed at USD 300.
3. Per hotel. Cheapest thresholds: Santiago 0.43, Viña del Mar 0.36, Puerto Varas 0.49.
4. Next busy week. Viña del Mar, week of 2025-02-10: about 24 cancellations expected; overselling
   20 rooms keeps the chance of walking anyone under 10%.
5. Caveats. The model ranks bookings well, not perfectly (AUC 0.74), and it overestimates the risk
   of bookings made more than four months ahead.


**Think about it.** Rewrite the memo in your own words in five lines at most. What would you cut? What would the manager ask that the memo does not answer?

<details><summary>Show the answer</summary>

One possible version:

*Resell the room of every booking whose predicted risk of cancelling is above 43%. On 1,491 past bookings this cuts the cost of cancellations by a quarter, about USD 48,000, at the price of walking 189 guests. Use a lower threshold at Viña del Mar and a higher one at Puerto Varas. For the week of 10 February in Viña, oversell 20 rooms. The model is less reliable for bookings made more than four months ahead: review those by hand until next month's improved model.*

The manager would also ask how walks will be handled (partner hotels?) and how we will know it works: propose a one-month pilot in one hotel.

</details>

## Graded quiz

Five questions on this week's ideas. They count for **5% of the module grade**, and **only your first submission counts**, so finish the lab before you start. Use the name and email you are registered with in the program.

<div class="hz-widget" data-widget="graded-quiz">
<script type="application/json">{"id": "session-4", "title": "Week 4 quiz", "questions": [{"q": "At threshold 0.5 a model flags 400 bookings as likely to cancel; 300 of them really cancel. In total 500 bookings cancelled. What are its precision and recall?", "options": ["Precision 0.75, recall 0.80", "Precision 0.75, recall 0.60", "Precision 0.60, recall 0.75", "Precision 0.80, recall 0.60"]}, {"q": "Altamar's model has 71% accuracy on the test bookings, where 66% of guests arrive. What is the best reading?", "options": ["The model is useless: it should be thrown away", "Accuracy barely beats \"nobody cancels\": judge the model by precision, recall, AUC and the cost of its mistakes", "71% accuracy means 71% of cancellations are caught", "The model is excellent: seven bookings in ten are right"]}, {"q": "In Puerto Varas a walked guest costs USD 450 and an empty room USD 550. In Santiago a walked guest costs USD 200 and an empty room USD 400. Where should the resale threshold be higher?", "options": ["In Santiago, because it has more bookings", "In Puerto Varas: its break-even, 450 / (450 + 550) = 0.45, is above Santiago's, 200 / (200 + 400) ≈ 0.33", "In Santiago, because its rooms are cheaper", "Nowhere: one model means one threshold"]}, {"q": "The 40 bookings for next week have cancellation probabilities that add up to 12.4. What can operations conclude?", "options": ["About 12 cancellations are expected; overselling 12 rooms would walk someone in roughly half of such weeks, so oversell fewer to be safe", "Nothing: probabilities cannot be added", "The 12 bookings with the highest probability will cancel", "Exactly 12 bookings will cancel"]}, {"q": "Bookings made 240 or more days ahead really cancel about 32% of the time, but the logistic model predicts 76%. What is the consequence, and the fix?", "options": ["We would leave too many rooms empty; raise the threshold", "We would resell too many of those rooms and walk guests; use a model whose curve can bend, such as a tree (week 5)", "The data is wrong; drop bookings made far ahead", "None: the average error over all bookings is small"]}]}</script>
</div>